[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/04_erro_prop.ipynb)

# Ajustamento Básico - A propagação dos erros

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Relacionar a propagação das variâncias aos modelos matemáticos lineares
2. Relacionar a propagação das variâncias aos modelos matemáticos não lineares
3. Exemplificar casos lineares e não lineares com aplicações da Eng. Cartográfica

**Referência:** Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. $\rightarrow$ **Cap 6, 7, 8 e 9**

## O Problema

Sabendo a precisão das observações, como obter a precisão do modelo/variáveis ajustadas?

$$A\underbrace{X}_{\text{parâmetros}} = \underbrace{L}_{\text{observações}} $$
$$ 
\underbrace{X_a}_{\text{modelo} \ (M)} = (A^T A)^{-1} A^T L
$$


## 1. Relacionar a propagação das variâncias aos modelos matemáticos lineares. 

Uma vez que todas as quantidades diretamente <b style="color:#2ECC40">observadas contêm erros</b>, quaisquer valores <b style="color:#2ECC40">calculados a partir delas também conterão erros</b>. Essa intrusão, ou propagação é um dos temas mais importantes discutidos nesta disciplina. 

A seguir, assume-se que todos os erros sistemáticos e equívocos foram eliminados das observações diretas, de modo que <b style="color:#CC2E40">apenas os erros aleatórios permaneçam</b>.

### **1.1 Antes de tudo, o que é variância?**

Resposta rápida: é o quadrado desvio padrão, ou seja, $\sigma^2$. 😑😑😑😑

A variância, assim como o desvio-padrão, é uma medida de <b style="color:#2ECC40">dispersão em relação à média</b>. Em termos algébricos:

$$
 \sigma_Z^2 = \frac{1}{gl} \sum_{i=1}^{m} (Z^i - \mu_{Z})^2 = E[( Z − \mu_Z)^2]
$$

onde:
- $Z$ é uma variável aleatória unidimensional (medida $m$ vezes); 
- $\mu_{Z}$ é sua média;
- $Z^i$ é um de seus componentes (uma medição); 
- $gl$ é o grau de liberdade do sistema (em se tratando de amostra, $gl=m-1$, em se tratando de população, $gl=m$, em se tratando de um sistema de $m$ equações e $u$ parâmetros, $gl=m-u$); e
- para fins de demonstrar a propagação dos erros, vamos utilizar a notação de operador linear de esperança $E$.

**broadcast**

$$\sigma_Z^2 = E[( Z − \mu_Z)^2]$$

Perceba dentro do operador esperança que, a princípio, $Z$ - vetorial - e $\mu_Z$ - escalar - não permitem a subtração. Está implícito o conceito de <b style="color:#2ECC40">broadcast</b> da álgebra vetorial e de *arrays* em computação, onde a grandeza escalar tem sua dimensionalidade aumentada, repetida tantas vezes quantas forem necessárias para permitir a operação elemento a elemento. Da mesma forma, o quadrado de $(Z -\mu_Z)$ - vetorial - é feito <b style="color:#2ECC40">element-wise</b>.

<center><img src="media/imgs/numpy_broadcasting.png" width=500></center>

### **1.2 Variância e desvio-padrão obtidos dos equipamentos**

Sob a ótica do ajustamento e em muitas aplicações, a variância não é calculada para
cada observação (isto seria caro!), mas retiradas das especificações técnicas do equipamento ou de posteriores
calibrações. 

<img src="media/imgs/img4.png" width=700>

### **1.3 O que é covariância?**

A covariância mensura a dependência linear entre 2 variáveis unidimensionais.

Sejam $Z'$ e $Z''$ duas variáveis com alguma relação de dependência entre si. A covariância entre elas, em notação estatística, é dada por:

$$
cov(Z',Z'') = \sigma_{Z',Z''} = E[(Z' - \mu_{Z'})(Z'' - \mu_{Z''})]
$$

Para o caso de <b style="color:#2ECC40">variáveis independentes, a covariância entre elas é nula</b>.


**Exercício 01:** Uma nuvem de pontos que representa um plano:

```python
num_points = 1000
x = np.random.uniform(-10, 10, num_points)
y = np.random.uniform(-10, 10, num_points)
z = (-1 * x - 2 * y - 5) / (-1)
```

In [2]:
import numpy as np
import open3d as o3d

# Equação do plano: ax + by + cz + d = 0
a, b, c, d = 1, 2, -1, 5  # Exemplo de plano qualquer
print(f"Gerando o plano {a}x + {b}y + {c}z + {d} = 0")

# Gerar pontos na nuvem
num_points = 1000
x = np.random.uniform(-10, 10, num_points)
y = np.random.uniform(-10, 10, num_points)
z = (-a * x - b * y - d) / c

# Criar array de pontos
points = np.column_stack((x, y, z))
print("XYZ:", points)

# Criar nuvem de pontos com Open3D
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Visualizar a nuvem de pontos
o3d.visualization.draw_plotly([pcd], width=400, height=300)

Gerando o plano 1x + 2y + -1z + 5 = 0
XYZ: [[  0.41941982   5.81066839  17.0407566 ]
 [  8.87333917   6.15115949  26.17565815]
 [  7.91800019   0.02645893  12.97091804]
 ...
 [ -5.94579246  -9.64637605 -20.23854455]
 [ -9.4140049   -2.68594402  -9.78589294]
 [  0.16231315   9.48730643  24.13692602]]


In [ ]:
# Covariâncias
cov_xy = np.cov(points[:, 0], points[:, 1])[0, 1]
cov_xz = np.cov(points[:, 0], points[:, 2])[0, 1]
cov_yz = np.cov(points[:, 1], points[:, 2])[0, 1]
print("Covariância de x com y:", cov_xy)
print("Covariância de x com z:", cov_xz)
print("Covariância de y com z:", cov_yz)

Covariância de x com y: 0.46602244702307816
Covariância de x com z: 33.19765247282787
Covariância de y com z: 67.51711111503963


Quais as variáveis independentes? 

Qual o grau de dependênica entre cada variável, comparando duas a duas?

**Exercício 02:** Uma nuvem de pontos que representa uma reconstrução 3D:

<img src="media/imgs/bunny.png">

In [4]:
import os
import wget
import open3d as o3d
import numpy as np

sample = "data/bunny.pcd"

if not os.path.exists(sample):
    url = 'https://raw.githubusercontent.com/PointCloudLibrary/pcl/master/test/bunny.pcd'
    print(f"Baixando o arquivo 'bunny.pcd' de {url}...")
    wget.download(url, sample)
    print("Download concluído.")

pcd = o3d.io.read_point_cloud(sample)
xyz = np.asarray(pcd.points)
print("XYZ:", xyz)

eixos = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.06, origin=[0,0,0])
o3d.visualization.draw_plotly([eixos,pcd], width=400, height=300)

XYZ: [[ 0.0054216  0.11349    0.040749 ]
 [-0.0017447  0.11425    0.041273 ]
 [-0.010661   0.11338    0.040916 ]
 ...
 [-0.064992   0.17802   -0.054645 ]
 [-0.069935   0.17983   -0.051988 ]
 [-0.07793    0.17516   -0.0444   ]]


In [ ]:
# Covariâncias
cov_xy = np.cov(xyz[:, 0], xyz[:, 1])[0, 1]
cov_xz = np.cov(xyz[:, 0], xyz[:, 2])[0, 1]
cov_yz = np.cov(xyz[:, 1], xyz[:, 2])[0, 1]
print("Covariância de x com y:", cov_xy)
print("Covariância de x com z:", cov_xz)
print("Covariância de y com z:", cov_yz)

Covariância de x com y: -0.0005930386397008904
Covariância de x com z: 0.00017801328678129232
Covariância de y com z: -0.0005568275720287393


Quais as variáveis independentes? 

Qual o grau de dependência entre cada variável, comparando duas a duas?

### **1.4 Matriz Variância-Covariância (MVC ou $\Sigma$)**

É o caso geral para se representar as variâncias e covariâncias de uma distribuição multivariada $Z = f(Z_1,...,Z_n)$ onde cada $Z_k$ representa uma variável aleatória unidimensional (medida $m$ vezes independentemente) com suas médias juntadas em um único vetor $U_Z = [\mu_{Z_1} \ ... \ \mu_{Z_n} ]^T$.

A MVC é definida como:

$$ \small
\mathrm{MVC}(Z) = \Sigma_{Z}
= \begin{bmatrix}
\sigma_{Z_1,Z_1} & \cdots & \sigma_{Z_1,Z_n} \\
\vdots &  \ddots & \vdots \\
\sigma_{Z_n,Z_1} & \cdots & \sigma_{Z_n,Z_n}
\end{bmatrix} $$
$$ \small = \begin{bmatrix}
E[(Z_1 - \mu_{Z_1})^2] & \cdots & E[(Z_1 - \mu_{Z_1})(Z_m - \mu_{Z_m})] \\
\vdots &  \ddots & \vdots \\
E[(Z_m - \mu_{Z_m})(Z_1 - \mu_{Z_1})]  & \cdots & E[(Z_m - \mu_{Z_m})^2]
\end{bmatrix} 
$$
$$ \small = E[(Z - U_Z)( Z - U_Z)^T]$$

onde:
- Elementos na posição $(i,j)$ da matriz representam covariâncias $\sigma_{Z_i,Z_j}$
- Com isso, a diagonal principal é composta pelas variâncias $\sigma^2_{Z_i}$ (notar que a VAR é um caso particular da COV com $i=j$)

### **1.5 Lei Geral de Propagação de Covariâncias**

Sejam duas variáveis multidimensionais $M$ e $L$ ligadas por um modelo linear:

$$
M = F(L) = GL + C
$$

onde:
- $L$: vetor com erros conhecidos com as seguintes médias e MVC:
    - $U_L = E[L]$
    - $\Sigma_{L} = E[(L - U_L)( L - U_L)^T]$
- $M$: vetor com erros desconhecidos;
- $G$: transformação linear; 
- $C$: vetor de constantes.

As incertezas em $L$ se propagam para $M$ através da transformação linear $G$ através da seguinte fórmula (demonstração na parte complementar final):

$$
\boxed{
\Sigma_{M} = \Sigma_{GL + C} = G\Sigma_{L} G^T
}
$$

Perceba que a <b style="color:#2ECC40">constante C não interfere na propagação dos erros</b>. Pode-se dizer que $C$ é determinístico, ou seja, sem incerteza própria.

**Exercício 03 (resolvido):** Seja $m = l_1 + l_2$, assumindo que $l_1$ e $l_2$ são valores observados independentemente, com desvios padrão $\sigma_1$ e $\sigma_2$ respectivamente, qual o desvio padrão de $m$?

**Solução exercício 03:**

$M = \underbrace{\begin{bmatrix}1 & 1\end{bmatrix}}_{G} \underbrace{\begin{bmatrix} l_1 \\ l_2\end{bmatrix}}_{L}$

$ \Sigma_l = \begin{bmatrix}cov(l_1, l_1) & cov(l_1, l_2) \\ cov(l_2, l_1) & cov(l_2, l_2) \end{bmatrix} = 
 \begin{bmatrix} \sigma^2_1 & 0 \\ 0 & \sigma^2_2 \end{bmatrix}$

$\Sigma_M = \begin{bmatrix}1 & 1 \end{bmatrix}\begin{bmatrix} \sigma^2_1 & 0 \\ 0 & \sigma^2_2 \end{bmatrix} \begin{bmatrix}1 \\ 1 \end{bmatrix} = \sigma^2_1 + \sigma^2_2 \leftarrow$ variância do modelo

$\sigma_m = \sqrt{\sigma^2_1 + \sigma^2_2} \leftarrow$ desvio padrão

> Nota: desvio padrão e erro padrão são termos equivalentes.

**Exercício 04:** Seja $\bar{y} =\frac{y_1 + y_2 + ... + y_n}{n}$, a média de obervações independentes $y_1, ..., y_n$, cada uma com o mesmo desvio padrão $\sigma$, qual o valor do erro/precisão/desvio padrão de $\bar{y}$?

$M = [\bar{y}] = \underbrace{\begin{bmatrix}1/n & ... & 1/n\end{bmatrix}}_{G} \underbrace{\begin{bmatrix} y_1 \\ \vdots \\ y_n\end{bmatrix}}_{L}$

$ \Sigma_l = \begin{bmatrix}cov(y_1, y_1) & cov(y_1, y_2) & ... \\ cov(y_2, y_1) & cov(y_2, y_2) & ...\\
\vdots & \ddots & \vdots \\ ... & ... & cov(y_n, y_n) \end{bmatrix} = 
 \begin{bmatrix} \sigma^2 & 0 & ... \\ 0 & \sigma^2 & ... \\ \vdots & \ddots & \vdots \\ 0 & 0 & \sigma^2 \end{bmatrix}$

$\Sigma_M =\begin{bmatrix}1/n & ... & 1/n\end{bmatrix}\begin{bmatrix} \sigma^2 & ... & 0\\ \vdots & \ddots & \vdots \\ 0 & ... & \sigma^2 \end{bmatrix} \begin{bmatrix}1/n \\ \vdots \\ 1/n\end{bmatrix} = \frac{1}{n^2} \left(\begin{bmatrix}\sigma^2 & ... & \sigma^2\end{bmatrix} \begin{bmatrix}1 \\ \vdots \\ 1\end{bmatrix}\right) = \frac{\sigma^2}{n}\leftarrow$ variância do modelo

$\sigma_m = \sigma_{\bar{y}}= \frac{\sigma}{\sqrt{n}} \leftarrow$ desvio padrão

**Exercício 05:** Seja o seguinte sistema linear com indicação de desvio padrão de cada resultado medido independentemente:

$$
\begin{cases}
3.00m + b = 4.5 \pm 0.01 \\
4.25m +b = 4.25 \pm 0.02 \\
5.50m + b = 5.5 \pm 0.03 \\
8.00m +b = 5.5 \pm 0.04
\end{cases}
$$

Através do ajustamento dos coeficientes de $m$ e $b$ pelo MMQ, calcule a precisão do modelo, ou seja, o erro padrão de $m$ e $b$ ajustados.

$M = X_a = \underbrace{(A^TA)^{-1} A^T}_{G} L$ , onde:

$X_a =  \begin{bmatrix}m \\ b  \end{bmatrix}$, $L = \begin{bmatrix}4.5 \\ 4.25 \\ 5.5 \\ 5.5 \end{bmatrix}$ e $A = \begin{bmatrix}3 & 1 \\ 4.25 & 1 \\ 5.5 & 1\\ 8 &1  \end{bmatrix}$

$ \Sigma_l = \begin{bmatrix}0.01^2 & & & 0 \\  & 0.02^2 & \\  &  & 0.03^2 \\  0 &  & & 0.04^2  \end{bmatrix}$

In [4]:
import numpy as np 
L = np.array([4.5 , 4.25 , 5.5 , 5.5 ])
A = np.array([[3 , 1],
              [4.25 , 1],
              [5.5 , 1],
              [8 ,1 ]])
G = np.linalg.inv(A.T @ A) @ A.T # @ L

Sigma_l =  np.array([[0.01**2 ,0,      0,      0],
                    [0,        0.02**2,0,      0],
                    [0,        0,      0.03**2,0],
                    [0,        0,      0,      0.04**2]])
Sigma_Xa = G @ Sigma_l @ G.T
print("Sigma_Xa:\n",Sigma_Xa)
sigma_m , sigma_b = Sigma_Xa[0,0], Sigma_Xa[1,1]
print("sigma^2_m:" ,sigma_m , "\nsigma^2_b:",sigma_b)
print("sigma_m:" ,sigma_m**0.5 , "\nsigma_b:",sigma_b**0.5)

Sigma_Xa:
 [[ 7.26204082e-05 -3.00146939e-04]
 [-3.00146939e-04  1.34729796e-03]]
sigma^2_m: 7.262040816326533e-05 
sigma^2_b: 0.0013472979591836747
sigma_m: 0.008521760860483315 
sigma_b: 0.036705557606221906


## 2. Propagação das variâncias para um modelo matemático não linear

### **2.1 Ferramentas para linearização - expansão em série Taylor**

Seja $Y = F(X)=F(X_0+\Delta X)$ o modelo não linear que liga $Y$ e $X$. 

- Usando expansão em série de Taylor, sendo $X_0$ uma estimativa aproximada de $X_a$:
$$
\small Y = F(X_0) + \frac{\partial F}{\partial X} \bigg|_{X=X_0} (X_a-X_0) + \frac{1}{2!} \frac{\partial^2F }{\partial X^2}(X_a-X_0)^2 + ...
$$

- Desprezando termos de ordem superior à primeira, temos um modelo linear(izado):

$$
\small Y \approx F(X_0) + \frac{\partial F}{\partial X} \bigg|_{X=X_0} (X_a-X_0)  \rightarrow \\  Y \approx F(X_0) + D \Delta X
$$



### **2.1 Ferramentas para linearização - expansão em série Taylor**

$$
\small Y = F(X) =F(X_0+\Delta X) \approx  F(X_0) + D \Delta X
$$

- Onde $D$ é a matriz Jacobiana:

$$
\small  D = \frac{\partial F}{\partial X} \bigg|_{X=X_0} = \begin{bmatrix}
\frac{\partial y_1}{\partial x_1} & \frac{\partial y_1}{\partial x_2}& ... & \frac{\partial y_1}{\partial x_n}\\
\frac{\partial y_2}{\partial x_1} & \frac{\partial y_2}{\partial x_2}& ... & \frac{\partial y_2}{\partial x_n}\\
... & ... & ... & ...    \\
\frac{\partial y_m}{\partial x_1} & \frac{\partial y_m}{\partial x_2}& ... & \frac{\partial y_m}{\partial x_n}\\
\end{bmatrix}
$$

### **2.2 A propagação da variância** 

Agora linearizado, com desenvolvimento similar, temos que incertezas em $X$ se propagam para $Y$ através da transformação linear $D$ através da seguinte fórmula:

$$
\boxed{
\Sigma_{Y} = \Sigma_{F(X_0) + D\Delta X}  = D\Sigma_{X_a}D^T
}
$$

> Nota: Mesmo que $\Sigma_X$ seja diagonal (observações independentes), $\Sigma_Y$ não será necessariamente diagonal. O modelo matemático interliga os parâmetros, o que correlaciona os mesmos.

> Nota: Na verdade, $\Sigma_{Y} = D\Sigma_{\Delta X}D^T$ mas $\Sigma_{\Delta X} = \Sigma_{X_a - X_0} = I\Sigma_{X_a}I^T=\Sigma_{X_a}$.


**Exercício 06 (resolvido):** Seja um tanque de dimensões $A \times B \times C$, com as seguintes medidas $A = 40 m \pm 0.05 m$, $B = 42 m \pm 0.03 m$ e $C = 15 m \pm 0.02 m$. Qual o valor do erro no volume usando estas observações?

**Solução exercício 06**

In [10]:
import sympy as sp
from IPython.display import display

# Definir as variáveis simbólicas
A, B, C = sp.symbols('A B C')
L = sp.Matrix([A, B, C])
sA, sB, sC = sp.symbols('s_A s_B s_C')

# Matriz de covariância da medição
Sigma_L = sp.Matrix([[sA**2, 0, 0], [0, sB**2, 0], [0, 0, sC**2]])
print("MVC observações (L=A,B,C)=")
display(Sigma_L)

# Volume
F = A*B*C

# Gradiente de F (Jacobian)
D = sp.Matrix([sp.diff(F, var) for var in L]).T
print("Matriz Jacobiana D =")
display(D)

# Cálculo da incerteza do volume usando a fórmula de propagação de erros
Sigma_F = D * Sigma_L * D.T
print("MVC do volume Sigma_F =")
display(Sigma_F)
sVol = sp.sqrt(Sigma_F[0])
print("Erro do volume:")
display(sVol)

# substituir os valores de A, B, C e as incertezas sA, sB, sC
A_val, B_val, C_val = 40, 42, 15
sA_val, sB_val, sC_val = 0.05, 0.03, 0.02
dados = {A: A_val, B: B_val, C: C_val, sA: sA_val, sB: sB_val, sC: sC_val}
print(f"Dados: {dados} ->")
sVol = float(sVol.subs(dados).evalf())
Vol = float(F.subs(dados).evalf())
print(f"Volume ± erro: {Vol} ± {sVol:.2f} m³")

MVC observações (L=A,B,C)=


Matrix([
[s_A**2,      0,      0],
[     0, s_B**2,      0],
[     0,      0, s_C**2]])

Matriz Jacobiana D =


Matrix([[B*C, A*C, A*B]])

MVC do volume Sigma_F =


Matrix([[A**2*B**2*s_C**2 + A**2*C**2*s_B**2 + B**2*C**2*s_A**2]])

Erro do volume:


sqrt(A**2*B**2*s_C**2 + A**2*C**2*s_B**2 + B**2*C**2*s_A**2)

Dados: {A: 40, B: 42, C: 15, s_A: 0.05, s_B: 0.03, s_C: 0.02} ->
Volume ± erro: 25200.0 ± 49.45 m³


### **2.3 Formulação geral da propagação dos erros**

Seja, $m = f(l_1, ..., l_n)$ uma **função não linear** de $n$ parâmetros $l_i$, medidos independentemente com variâncias respectivamente $\sigma^2_{l_1},..., \sigma^2_{l_n}$

$$
\Sigma_m = D \Sigma_L D^T = 
\begin{bmatrix}
\frac{\partial m}{\partial l_1} & ... & \frac{\partial m}{\partial l_n}
\end{bmatrix}
\begin{bmatrix}
\sigma^2_{l_1} & & 0\\ 
 & \ddots & \\
0 & & \sigma^2_{l_n}
\end{bmatrix}
\begin{bmatrix}
\frac{\partial m}{\partial l_1} \\ \vdots \\ \frac{\partial m}{\partial l_n}
\end{bmatrix}=
\left[\left(\frac{\partial m}{\partial l_1}\right)^2\sigma^2_{l_1} + ... + \left(\frac{\partial m}{\partial l_n}\right)^2\sigma^2_{l_n}\right]
$$
$$
\boxed{\sigma_m = \sqrt{\left(\frac{\partial m}{\partial l_1}\right)^2\sigma^2_{l_1} + ... + \left(\frac{\partial m}{\partial l_n}\right)^2\sigma^2_{l_n}}}
$$

### **2.4 A propagação da variância em modelos matemáticos não lineares após o processo iterativo de Gauss Newton**

Durante o processo iterativo de Gauss-Newton, os parâmetros são ajustados ao longo das iterações pela fórmula: 

$\underbrace{X_a}_{\substack{\text{erro}\\ \text{desconhecido}}} = -(J^TJ)^{-1}J^T\underbrace{L}_{\substack{\text{erro}\\ \text{conhecido}}} + X_0$

**1ª parte**, de $\Sigma_{L_b}$ para $\Sigma_{L}$:

$$L = \underbrace{L_0}_{\text{determinístico}}-L_b \rightarrow \Sigma_{L}=(-I)\Sigma_{L_b}(-I)^T=\Sigma_{L_b}$$

**2ª parte**, de $\Sigma_{L}$ para $\Sigma_{\Delta X}$:

$$\Delta X = -(J^TJ)^{-1}J^TL \rightarrow \Sigma_{\Delta X}=-I(J^TJ)^{-1}J^T\Sigma_{L}(-I(J^TJ)^{-1}J^T)^T $$

Ou seja, para a $k$-ésima iteração, apenas muda o ponto onde $J$ é avaliado:

$$\Sigma_{\Delta X_k}=(J_k^TJ_k)^{-1}J_k^T\Sigma_{L_b}J_k(J_k^TJ_k)^{-1} $$

**3ª parte**, de $\Sigma_{\Delta X}$ para $\Sigma_{X_a}$:

$$X_a = \underbrace{X_0}_{\text{fixo na iteração}} + \Delta X \rightarrow \Sigma_{X_a} = \Sigma_{\Delta X}$$

**Ao final**, 

$$\boxed{\Sigma_{X_a} = (J^TJ)^{-1}J^T\Sigma_{L_b}J(J^TJ)^{-1} \text{, para } J \text{ avaliado em } X_a}$$

## 3. Aplicações práticas da lei de propagação das variâncias na Eng. Cartográfica

### **3.1 Exemplos de casos lineares**

**Conversão entre SRC planos** - Transformação afim 2D de escalonamento e translação:

<img src="media/imgs/img8.jpg">

A formulação da conversão das coordenadas planas é:

$$
\underbrace{\begin{bmatrix}
x'\\y'
\end{bmatrix}}_{X'} =
\begin{bmatrix}
s_x & 0 \\ 0 & s_y
\end{bmatrix}
\underbrace{\begin{bmatrix}
x \\ y
\end{bmatrix}}_{X} + \begin{bmatrix}
t_x \\ t_y
\end{bmatrix}
$$

**Conversão entre SRC planos** 

Em uma conversão de coordenadas com modelo estabelecido, a MVC das coordenadas no novo referencial é:

$$
\underbrace{\begin{bmatrix}
x'\\y'
\end{bmatrix}}_{X'} =
\underbrace{\begin{bmatrix}
s_x & 0 \\ 0 & s_y
\end{bmatrix}}_{G}
\underbrace{\begin{bmatrix}
x \\ y
\end{bmatrix}}_{X} + \underbrace{\begin{bmatrix}
t_x \\ t_y
\end{bmatrix}}_{cte} \rightarrow \Sigma_{X'} = G\Sigma_{X} G^T
$$

**Conversão entre SRC planos** 

Em um ajustamento do modelo, a MVC dos parâmetros ajustados é:

$$
\underbrace{
\begin{bmatrix}
x'_1\\y'_1
\end{bmatrix}}_{L} =
\underbrace{
\begin{bmatrix}
x_1  & 1 & 0 & 0 \\
0  & 0 & y_1 & 1 \\
\vdots & \vdots & \vdots & \vdots
\end{bmatrix}}_{A} \underbrace{\begin{bmatrix}
s_x \\ t_x \\ s_y \\ t_y
\end{bmatrix}}_{X} \rightarrow  \text{2 Eq. por ponto homólogo } (X'_i, X_i) $$

$$ X_a = \underbrace{(A^TA)^{-1} A^T}_{G} L \rightarrow \Sigma_{X_a} = G\Sigma_{L} G^T$$


**Fechamento de poligonal** - Assumindo que cada ângulo da poligonal foi observado 4 vezes (2 vante e 2 ré) e seus erros estimados conforme tabela abaixo:

<center><img src="media/imgs/img10.jpeg"></center>

O erro do somatório dos ângulos internos desse polígono é:

$$y = a_1 + a_2 + a_3 + a_4 + a_5 = 540^o30''$$
$$
\sigma_y = \sqrt{\left(\frac{\partial y}{\partial a_1}\right)^2\sigma^2_{1} + ... + \left(\frac{\partial y}{\partial a_5}\right)^2\sigma^2_{5}} = \sqrt{\sigma^2_{1} + ... + \sigma^2_{5}} = 24.7''
$$

<!-- > Nota: **(Assunto de ajustamento avançado)** - para ter o fechamento angular aceito, consideramos o erro como uma distribuição-*t*, a soma angular observada, considerando o erro, tem que que satisfazer a condição $S_n = (n-2) \times 180^o$ para $n=5$, dentro de um determinado intervalo de confiança $\alpha$: $y - t_{(\alpha/2, gl)}\sigma_y < S_5 < y + t_{\alpha/2, gl}\sigma_y ?$ -->


### **3.1 Exemplos de casos não lineares**

**Trilateração** - Intersecção por distâncias é a base geométrica do GPS. 

<center><img src="media/imgs/img12.png" width=600></center>

A formulação do sistema de equações redundante mínima (4 observações) é:
    
$$\begin{cases}
d_1 = ||P-S_1|| \ \ \rightarrow (f_1) \\
d_2 = ||P-S_2|| \ \ \rightarrow (f_2) \\
d_3 = ||P-S_3|| \ \ \rightarrow (f_3) \\ 
d_4 = ||P-S_4|| \ \ \rightarrow (f_4) \\ 
\end{cases}$$

Para se encaixar no MMQ não linear teremos:

$$X_a = \begin{bmatrix} X_p \\  Y_p \\ Z_p \end{bmatrix}$$

$$
F(X_a) = \begin{bmatrix} f_1(X_a) \\  f_2(X_a) \\ f_3(X_a) \\ f_4(X_a) \end{bmatrix} = 
\begin{bmatrix}
    ||P-S_1|| \\
    ||P-S_2|| \\
    ||P-S_3|| \\
    ||P-S_4||
\end{bmatrix}$$

$$
L_b+V = \begin{bmatrix}
d_1 + v_1 \\
d_2 + v_2 \\
d_3 + v_3 \\
d_4 + V_4
\end{bmatrix}
$$

$$
\Sigma_{L_b} = \begin{bmatrix}
\sigma_1 & & & 0 \\
 & \sigma_2 & &   \\
  & & \sigma_3 &   \\
0 & & & \sigma_4 
\end{bmatrix}
$$

Após a solução iterativa, o erro nas coordenadas do alvo será dado por:

$$
\Sigma_{X_a} = (J^TJ)^{-1}J^T\Sigma_{L_b}J(J^TJ)^{-1} \text{ para } J \text{ avaliado em } X_a
$$


**Nivelamento Trigonométrico**

<center><img src="media/imgs/img9.jpeg" height=200></center>

A formulação do desnível entre um instrumento de medição (estação total) e um prisma é:

$$
\Delta h = h_i + S sen(z) + h_{CR} − h_r 
$$

onde:
- $h_i$ e $h_r$ são a altura do instrumento e do prisma (*rod*) em relação ao chão;
- $S$ e $v$ são a distância e o ângulo vertical entre o instrumento e o prisma;
- O ângulo prisma-instrumento-zênite é dado por $z=90^o-v$; e
- A correção da curvatura e refração terrestre é dada por $h_{CR} = CR(S sen(z) 10^{-3})^2$.

## Complemento: Demonstração da lei de propagação das covariâncias

$$
\Sigma_{M} = E[(M - U_M)(M - U_M)^T] \xrightarrow{?}
\boxed{
\Sigma_{M} = \Sigma_{GL + C} = G\Sigma_{L} G^T
}
$$

**1ª passo**: uso das propriedades lineares do operador esperança em $\small U_M = E[GL + C]$

- Pela propriedade da aditividade de operadores lineares:

$\small U_M = E[GL] + E[C]$

- O operador esperança não atua em constantes, a esperança de uma constante é ela própria:

$\small E[C] = C$

- Pela propriedade da homogeneidade (cada $g_{ij}$ é constante, não aleatório):

$\small E[GL] \rightarrow \text{linha } i=\sum_{j=1}^{n} E[g_{ij}L_j] = \sum_{j=1}^{n} g_{ij}E[L_j] =G_iE[L] \rightarrow​ E[GL] = GE[L]$

- Assim:

$\small U_M = G U_L + C$

**2ª passo**: uso da equação e propriedades do operador linear esperança acima na definição de MVC em $\Sigma_{M}$

$$\small 
\begin{align*}
\Sigma_{M} &= E[(M - U_M)(M - U_M)^T]  \\
 & = E[(GL + C - G U_L - C)(GL + C - G U_L - C)^T] \\
 & = E[(G(L - U_L))(G(L-U_L))^T] \\
 & = E[G(L - U_L)(L-U_L)^TG^T] \\
 & = GE[(L - U_L)(L-U_L)^T]G^T \\
 & = G\Sigma_{L}G^T
 \end{align*}
$$

## Lista de exercícios complementares

**Exercício 07**

Sejam L1=[232,5 232,7 232,5 232,4 232,4] e L2=[72,8 72,6 72,5 72,4 72,7] as medidas de comprimento e largura de um terreno retangular em metros. Seja X=[X1 X2], com X1=diagonal do retângulo e X2=área do retângulo. Calcule Σx.

In [1]:
import numpy as np

L1=np.array([232.5, 232.7, 232.5, 232.4, 232.4])
L2=np.array([72.8, 72.6, 72.5, 72.4, 72.7])

# https://numpy.org/doc/stable/reference/generated/numpy.std.html#numpy.std
# ddof=1 means we are calculating the sample standard deviation, 
# which is appropriate when estimating the standard deviation from a sample 
# rather than the entire population. 
print("l1 =" , float(L1.mean()), "+-", float(L1.std(ddof=1)))
print("l2 =",  float(L2.mean()), "+-", float(L2.std(ddof=1)))


l1 = 232.5 +- 0.12247448713915195
l2 = 72.6 +- 0.15811388300841672


Considerando erros aleatórios o sistema de equações é:

$\begin{cases} x_1 = \sqrt{l_1^2 + l_2^2}\\ x_2 = l_1 l_2\end{cases}$

$\Sigma_L = \begin{bmatrix}var(l_1) & cov(l_1,l_2) \\ cov(l_2,l_1) & var(l_2)\end{bmatrix}$

$cov(l_1,l_2) = cov(l_2,l_1) \ne 0 $ pois não foi falado que são observações independentes entre si.

> A pergunta se traduz em descobrir como os erros nas variáveis $l_1$ e $l_2$ se propagam para as variáveis $x_1$ e $x_2$.

Teste se possível usar `np.cov` para obtenção da MVC(L) com graus de liberdade `n-1`:

In [2]:
import sympy as sp
from IPython.display import display

Sigma_L = np.cov([L1,L2])
print("Sigma_L=\n",Sigma_L)

# checagem, se usando degrees of freedom n-1:
mean = sum(L1) / len(L1)
squared_diffs = [(l - mean) ** 2 for l in L1]
var_l1 = sum(squared_diffs) / (len(L1) - 1)
print(f"\nvar(l1) = {var_l1}")

f"var_l1 = Sigma_L[0,0] ?"


Sigma_L=
 [[0.015  0.0025]
 [0.0025 0.025 ]]

var(l1) = 0.014999999999998296


'var_l1 = Sigma_L[0,0] ?'

Solução:

In [4]:
SigmaL_symbol

Covariance(l_1, l_2)

In [9]:
from sympy.stats import Covariance
from IPython.display import Latex

l1 = sp.Symbol("l_1")  
l2 = sp.Symbol("l_2")  

SigmaL_symbol = sp.Matrix([[Covariance(l1, l1), Covariance(l1, l2)], 
                            [Covariance(l2, l1), Covariance(l2, l2)]])

x1 = sp.sqrt(l1**2 + l2**2)
x2 = l1 * l2

J = sp.Matrix([[sp.diff(x1,l1),sp.diff(x1,l2)],
               [sp.diff(x2,l1),sp.diff(x2,l2)]])
print("Jacobiana simbólica (J):")
display(J)
print("Jacobiana avaliada nas observações ajustadas:")
display(J.subs({l1:L1.mean(), l2: L2.mean()}).evalf())

display(Latex(r"""Solução simbólica: 
$\Sigma_X = J \Sigma_L J^T=$
"""))
Sigma_X = J * SigmaL_symbol * J.T
display(Sigma_X)

print("Solução com Jacobiana avaliada nas observações ajustadas (Σx):")
J = J.subs({l1:L1.mean(), l2: L2.mean()}).evalf()
SigmaX = J * Sigma_L * J.T
display(SigmaX)


Jacobiana simbólica (J):


Matrix([
[l_1/sqrt(l_1**2 + l_2**2), l_2/sqrt(l_1**2 + l_2**2)],
[                      l_2,                       l_1]])

Jacobiana avaliada nas observações ajustadas:


Matrix([
[0.954545702455981, 0.298064593541093],
[             72.6,             232.5]])

<IPython.core.display.Latex object>

Matrix([
[l_1*(l_1*Covariance(l_1, l_1)/sqrt(l_1**2 + l_2**2) + l_2*Covariance(l_1, l_2)/sqrt(l_1**2 + l_2**2))/sqrt(l_1**2 + l_2**2) + l_2*(l_1*Covariance(l_1, l_2)/sqrt(l_1**2 + l_2**2) + l_2*Covariance(l_2, l_2)/sqrt(l_1**2 + l_2**2))/sqrt(l_1**2 + l_2**2), l_1*(l_1*Covariance(l_1, l_2)/sqrt(l_1**2 + l_2**2) + l_2*Covariance(l_2, l_2)/sqrt(l_1**2 + l_2**2)) + l_2*(l_1*Covariance(l_1, l_1)/sqrt(l_1**2 + l_2**2) + l_2*Covariance(l_1, l_2)/sqrt(l_1**2 + l_2**2))],
[                                                                                        l_1*(l_1*Covariance(l_1, l_2) + l_2*Covariance(l_1, l_1))/sqrt(l_1**2 + l_2**2) + l_2*(l_1*Covariance(l_2, l_2) + l_2*Covariance(l_1, l_2))/sqrt(l_1**2 + l_2**2),                                                                                         l_1*(l_1*Covariance(l_2, l_2) + l_2*Covariance(l_1, l_2)) + l_2*(l_1*Covariance(l_1, l_2) + l_2*Covariance(l_1, l_1))]])

Solução com Jacobiana avaliada nas observações ajustadas (Σx):


Matrix([
[0.0173110064033209, 3.38092913321209],
[  3.38092913321209, 1514.86514999993]])

**Exercício 08**

A distância do ponto C às estações A e B deve ser determinada a partir das medidas dos ângulos A e B e do lado c. Os lados a, b e c são opostos aos ângulos A, B e C, respectivamente. A média das observações foi L=[A B c]=[90º 45º 100m]; σA=σB=4’’;
σc=5cm; σAB=1’’; demais correlações são nulas. Calcule X=[a b] e Σx.


In [ ]:
# code here

**Exercício 09**

Uma distância de aproximadamente 500m (D) deve ser medida com fita de 50m de comprimento. O desvio-padrão máximo tolerado em D é σD=10mm. Determine qual a precisão mínima necessária para cada lance de fita. Admitir que os lances serão de igual
precisão (σ).

In [ ]:
# code here

**Exercício 10**

A distância inclinada entre os pontos A, no terreno, e B, no alto de uma torre, deve ser determinada com precisão máxima de σD=20mm. Sabe-se que a distância horizontal dentre eles é de 400m e tem erro desprezível e que a altura da torre é de aproximadamente 80m. Qual precisão máxima com a qual a altura deve ser medida para assegurar o σD especificado.


In [ ]:
# code here

**Exercício 11**

Para determinar o lado c de um triângulo, os lados a e b e o ângulo C foram medidos. Os lados a, b e c são opostos aos ângulos A, B e C, respectivamente. Dados: a=30m±3mm; b=40m±4mm; C=60°±5”; 𝜎𝑎𝑏= 𝜎𝑎𝐶=𝜎𝑏𝐶=0. Estime o comprimento do lado c e sua precisão σC

In [ ]:
# code here

**Exercício 12**

No **nivelamento trigonométrico**, identifique as observações com erros, encaixe o problema na abordagem MMQ não linear. Como os erros se propagam até o desnível calculado?


## Lista de exercícios suplementares

Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. 

Pág 138 a 141 (7. ERROR PROPAGATION IN ANGLE AND DISTANCE OBSERVATIONS - Problems).


<!-- **Exercício 4 gabarito:**

$\bar{y} = \underbrace{\frac{1}{n} \begin{bmatrix} 1 & 1 & ... & 1\end{bmatrix}}_{G} \underbrace{\begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_n\end{bmatrix}}_{L}$

$\Sigma_{L} = \begin{bmatrix}cov(y_1, y_1) & ... & cov(y_1, l_n) \\ \vdots & \ddots & \vdots \\ cov(l_n, l_1) & ... & cov(l_n, l_n) \end{bmatrix} = \begin{bmatrix}\sigma^2 &  & 0 \\  & \ddots & \\ 0 & & \sigma^2 \end{bmatrix} = \sigma^2 I$

$\Sigma_{\bar{y}} = G \Sigma_{L} G^T = \frac{1}{n} \begin{bmatrix} 1 & 1 & ... & 1\end{bmatrix} \sigma^2 I  \frac{1}{n} \begin{bmatrix} 1 \\ 1 \\ \vdots \\ 1\end{bmatrix} = \frac{\sigma^2}{n^2}\begin{bmatrix} 1 & 1 & ... & 1\end{bmatrix} \begin{bmatrix} 1 \\ 1 \\ \vdots \\ 1\end{bmatrix} = \frac{\sigma^2}{n}$

$\sigma_{\bar{y}}= \sqrt{\frac{\sigma^2}{n}} = \frac{\sigma}{\sqrt{n}}$ -->

<!-- **Exercício 5 gabarito:**

$\Sigma_{L} = \begin{bmatrix} 0.01^2 & 0.02^2 & 0.03^2 & 0.04^2 \end{bmatrix}I $

$X_a = \underbrace{(A^TA)^{-1} A^T}_{G} \ L$

$\Sigma_{X_a} = \Sigma_{ G L} = G\Sigma_{L} G^T $

$\sigma_m = \sqrt{\Sigma_{X_a}[1,1]} \ \ $ e $\ \ \sigma_b = \sqrt{\Sigma_{X_a}[2,2]}$

```python
import numpy as np

A = np.array([[3.0, 1],
              [4.25, 1],
              [5.5, 1],
              [8.0, 1]])
Sigma_L = np.diag([0.01**2, 0.02**2, 0.03**2, 0.04**2])  # Matriz de covariância das medições
G = np.linalg.inv(A.T @ A) @ A.T
Sigma_X = G @ Sigma_L @ G.T
print("MVC(X_a): Sigma_X)

var_m = Sigma_X[0, 0]
var_b = Sigma_X[1, 1]
print("Erro do parâmetro m: var_m**0.5)
print("Erro do parâmetro b: var_b**0.5)
```
 -->